# 05 · 장애 진단과 복구 확인

목표: 상태→영향→Trace→완화→재검증 순서로 장애 기록을 만든다. 약15분.

**실행:** Python 3.11 공통 .venv 커널에서 위→아래. 다른 Notebook·서버·Docker·모델 설치 없이 독립 실행. 코드는 MOCK 고정 문장·대기만 사용한다. 실제 LLM 품질·처리량 측정이 아니다.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week10/lab/app.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from week10.lab.experiments import lab_client, fixture_check

## 준비 상태와 생존 상태

의존성 장애에서 프로세스를 무조건 재시작하는 것이 적절한지 토론한다.

In [ ]:
async with lab_client() as (app, client):
    app.state.dependency_ok = False  # test fixture; production control API 아님
    live = await client.get("/health/live")
    ready = await client.get("/health/ready")
    blocked = await client.post("/api/v1/assist", json={"message": "예시"})
assert [live.status_code, ready.status_code, blocked.status_code] == [200, 503, 503]

## 시간 초과의 원인

정상과 지연 조건을 한 앱에서 비교한다.

In [ ]:
async with lab_client() as (app, client):
    normal = await client.post("/api/v1/assist", json={"message": "예시"})
    failed = await client.post("/api/v1/assist", json={"message": "예시", "scenario": "slow"})
    snap = app.state.telemetry.snapshot()
assert failed.status_code == 504 and snap["active"] == 0

## 실패 Trace 조회

HTTP 상태만으로 검색 장애·생성 지연을 같은 원인으로 결론내리지 않는다.

In [ ]:
tid = failed.headers["X-Trace-ID"]
fault_trace = [s for s in snap["spans"] if s["trace_id"] == tid]
[(s["name"], s["duration_ms"], s["status"]) for s in fault_trace]

## 복구 실험

주입 조건을 normal로 돌리고 이후 요청의 성공과 active 해제를 함께 본다.

In [ ]:
async with lab_client() as (app, client):
    await client.post("/api/v1/assist", json={"message": "예시", "scenario": "slow"})
    recovered = await client.post("/api/v1/assist", json={"message": "예시"})
    assert recovered.status_code == 200 and app.state.telemetry.active == 0
    print(recovered.json())

## 수신 중단과 처리 종료

이 앱은 비스트리밍이다. 연결 종료만으로 외부 모델 계산이 멈춘다고 가정하지 않는다.

In [ ]:
print("이 실험의 제공자: 고정 문장+asyncio.sleep")
print("외부 GPU 취소: 미구현·미측정")

## 상태 복원

의존성 fixture 복원 후 ready·정상 응답·기준 통과를 확인한다.

In [ ]:
async with lab_client() as (app, client):
    app.state.dependency_ok = False
    assert (await client.get("/health/ready")).status_code == 503
    app.state.dependency_ok = True
    assert (await client.get("/health/ready")).status_code == 200
    r = await client.post("/api/v1/assist", json={"message": "예시"})
    assert fixture_check(r.json()["answer"])

## 장애 기록

실측·가설·실험실 조작을 구분한다. 재시작은 영구 해결의 증거가 아니다.

In [ ]:
incident = {
    "symptom": "504",
    "evidence_trace": tid,
    "hypothesis": "generate 지연",
    "mitigation": "주입 조건 normal 복원",
    "verification": "후속200 + 기준통과 + active0",
    "remaining": "실제 제공자/장기 관찰 미검증",
}
incident

## 프로젝트 적용 기록

예상 → 관찰 → 근거 → 다음 변경을 한 줄씩 기록한다. 실제로 실행하지 않은 조건은 미측정으로 표시한다. 학생 정보나 원문 입력은 공개 저장소에 올리지 않는다.